#### Loading cleaned datasets

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ECB API data
df = pd.read_csv('inflation_clean.csv')

# Eurostat API
df2 = pd.read_csv('interest_clean.csv')

# Energy clean
df3 = pd.read_csv('energy_clean.csv')

# Energy_clean_index
df4 = pd.read_csv('energy_clean_index.csv')

df.head(25)

,date,country,inflation
0,2015-01,DE,-0.5
1,2015-02,DE,-0.2
2,2015-03,DE,0.3
3,2015-04,DE,1.0
4,2015-05,DE,1.6
5,2015-06,DE,1.1
6,2015-07,DE,1.3
7,2015-08,DE,1.1
8,2015-09,DE,0.8
9,2015-10,DE,1.2


In [47]:
df2.head()

,date,interest
0,2015-12,-0.30
1,2016-03,-0.40
2,2019-09,-0.50
3,2022-07,0.00
4,2022-09,0.75


In [48]:
df3.head(5)

,date,country,energy
0,2018-01,DE,106.35
1,2018-02,DE,106.22
2,2018-03,DE,106.49
3,2018-04,DE,106.51
4,2018-05,DE,106.65


In [49]:
df4.head(4)

,date,country,energy_price_index
0,2015-01-01,DE,100.5
1,2015-02-01,DE,101.9
2,2015-03-01,DE,101.9
3,2015-04-01,DE,101.5


---

In [50]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df2["date"] = pd.to_datetime(df2["date"], errors="coerce")

In [51]:
df2["interest"] = pd.to_numeric(df2["interest"], errors="coerce")

# Keep only what we need
df2 = df2[["date", "interest"]]

# Remove the duplicates
df2 = df2.drop_duplicates(subset=["date"])
df = df.drop_duplicates(subset=["country", "date"])


### Merging (inflation and interest) the datasets via date
---

In [52]:
# Making sure the date format matches each dataset
df['date'] = pd.to_datetime(df['date']).dt.to_period('M').astype(str)
df2['date'] = pd.to_datetime(df2['date']).dt.to_period('M').astype(str)

In [53]:
# Merge the two datsets
df_merged = pd.merge(df,df2, on='date', how='left')

# Sort the values
df_merged = df_merged.sort_values(['country', 'date'])

In [54]:
df_merged

,date,country,inflation,interest
0,2015-01,DE,-0.5,NaN
1,2015-02,DE,-0.2,NaN
2,2015-03,DE,0.3,NaN
3,2015-04,DE,1.0,NaN
4,2015-05,DE,1.6,NaN
...,...,...,...,...
427,2023-08,IT,5.5,3.75
428,2023-09,IT,5.6,4.00
429,2023-10,IT,1.8,NaN
430,2023-11,IT,0.6,NaN


In [55]:
# Filling in the Nan values
df_merged = df_merged.sort_values(["country", "date"])

df_merged["interest"] = df_merged["interest"].ffill().bfill()

In [56]:
df_merged["date"] = pd.to_datetime(df_merged["date"]).dt.to_period("M")


In [57]:
df_merged.tail()

,date,country,inflation,interest
427,2023-08,IT,5.5,3.75
428,2023-09,IT,5.6,4.00
429,2023-10,IT,1.8,4.00
430,2023-11,IT,0.6,4.00
431,2023-12,IT,0.5,4.00


In [58]:
df_merged.shape

(432, 4)

In [59]:
df_merged['country'].unique()

array(['DE', 'ES', 'FR', 'IT'], dtype=object)

In [60]:
print(df_merged[df_merged["inflation"].isna()].head(20))

Empty DataFrame
Columns: [date, country, inflation, interest]
Index: []


In [61]:
df_merged.head(5)

,date,country,inflation,interest
0,2015-01,DE,-0.5,-0.3
1,2015-02,DE,-0.2,-0.3
2,2015-03,DE,0.3,-0.3
3,2015-04,DE,1.0,-0.3
4,2015-05,DE,1.6,-0.3


#### Merge (energy_clean and energy_clean)index

In [62]:
df3.head()

,date,country,energy
0,2018-01,DE,106.35
1,2018-02,DE,106.22
2,2018-03,DE,106.49
3,2018-04,DE,106.51
4,2018-05,DE,106.65


In [63]:
df4.head()

,date,country,energy_price_index
0,2015-01-01,DE,100.5
1,2015-02-01,DE,101.9
2,2015-03-01,DE,101.9
3,2015-04-01,DE,101.5
4,2015-05-01,DE,101.8


In [64]:
# Fixing date columns -- make sure they are the same
df3["date"] = pd.to_datetime(df3["date"]).dt.to_period("M")
df4["date"] = pd.to_datetime(df4["date"]).dt.to_period("M")


In [65]:
df_energy = pd.merge(df3, df4, on=['date', 'country'], how='inner')

In [66]:
df_energy

,date,country,energy,energy_price_index
0,2018-01,DE,106.35,98.4
1,2018-02,DE,106.22,97.7
2,2018-03,DE,106.49,97.9
3,2018-04,DE,106.51,98.6
4,2018-05,DE,106.65,99.6
...,...,...,...,...
7195,2023-08,IT,9.20,175.2
7196,2023-09,IT,8.80,175.6
7197,2023-10,IT,7.90,176.8
7198,2023-11,IT,6.90,173.1


### Merge to create final dataframe
---

In [ ]:
final_df = pd.merge(df_merged,df_energy, on=["date", "country"], how="inner")

In [70]:
final_df.head()

,date,country,inflation,interest,energy,energy_price_index
0,2018-01,DE,1.5,-0.4,106.35,98.4
1,2018-01,DE,1.5,-0.4,106.71,98.4
2,2018-01,DE,1.5,-0.4,102.24,98.4
3,2018-01,DE,1.5,-0.4,99.93,98.4
4,2018-01,DE,1.5,-0.4,101.67,98.4


In [71]:
final_df.shape

(7200, 6)

# Things to do next???

1. Fix any dups
2. Check missing values
3. Validate ranges for finak check 
4. Create column about green deal
5. Do we need any more columns for our business case???
5. Visuals 



#### Visuals
 ---- 